In [1]:
# Imports
import numpy as np
import calc_cld
import importlib
import pingouin as pg

In [32]:
importlib.reload(calc_cld);

In [14]:
# Load some test data#
penguins = pg.read_dataset("penguins")
penguins_tk_results = penguins.pairwise_tukey(dv='body_mass_g', between='species')
penguins_tk_results

,A,B,mean(A),mean(B),diff,se,T,p-tukey,hedges
0,Adelie,Chinstrap,3700.662252,3733.088235,-32.425984,67.511684,-0.480302,0.880667,-0.073946
1,Adelie,Gentoo,3700.662252,5076.016260,-1375.354009,56.147971,-24.495169,0.000000,-2.860201
2,Chinstrap,Gentoo,3733.088235,5076.016260,-1342.928025,69.856928,-19.223978,0.000000,-2.875327


In [52]:
#
def determine_letters(letter_matrix, unique_groups, letter_type='low_a-z'):
    cld_dict = {}
    
    # How many letters a needed in total.
    n_letters = letter_matrix.shape[1]
    # Define which letters to use. (Doing it this way offers more flexibility later.)
    if letter_type == 'low_a-z':
        letters = [chr(i) for i in range(97, 97 + n_letters)]  # a-z
    elif letter_type == 'up_A-Z':
        letters = [chr(i) for i in range(65, 65 + n_letters)]  # A-Z
    else:
        raise ValueError("Not a valid letter type was chosen. Choose 'low_a-z' or 'up_A-Z'.")
    # Assign letters to groups.
    for i, group in enumerate(unique_groups):
        group_letters = ''
        for j in range(n_letters):
            if letter_matrix[i, j] == 1:
                group_letters += letters[j]
        cld_dict[group] = group_letters

    return cld_dict

def fill_all_zero_rows(letter_matrix):
    # Check for any all-0 rows
    zero_rows = np.all(letter_matrix == 0, axis=1)
    any_row_all_zero = np.any(zero_rows)
    if any_row_all_zero:
        # Create a new matrix with an additional column
        new_matrix = np.zeros((letter_matrix.shape[0], letter_matrix.shape[1] + 1), dtype=np.int8)
        # Copy the old matrix into the new one
        new_matrix[:, :-1] = letter_matrix
        # Set the last column to 1 for all-0 rows
        new_matrix[zero_rows, -1] = 1
        return new_matrix
    else:
        return letter_matrix

def run_clc(group_1_col, group_2_col, pvals, alpha=0.05):
    # 1) Build capital_H.
    capital_H = calc_cld.calc_big_H(group_1_col, group_2_col, pvals, alpha)
    # 2) List unique groups.
    unique_groups = calc_cld.list_unique_groups(group_1_col, group_2_col)
    # 3) Insert and absorb
    letter_matrix = calc_cld.heuristic_insert_absorb(unique_groups, capital_H)
    # 4) Sweep 
    letter_matrix = calc_cld.sweep(letter_matrix)
    # Fill in any all-0 rows
    letter_matrix = fill_all_zero_rows(letter_matrix)
    # 5) Determine letters
    final_letters = determine_letters(letter_matrix, unique_groups)
    # 6) Verify that the calculated solutions solves the problem correctly.
    is_valid = calc_cld.verify_cld(final_letters, group_1_col, group_2_col, pvals, alpha)
    print(f"CLD valid: {is_valid}")
    return final_letters

group_1_col, group_2_col, pvals = list(penguins_tk_results['A']), list(penguins_tk_results['B']), list(penguins_tk_results['p-tukey'])

final_letters = run_clc(group_1_col, group_2_col, pvals)
print(final_letters)

CLD valid: True
{'Adelie': 'a', 'Chinstrap': 'a', 'Gentoo': 'b'}


In [51]:
penguins_tk_results

,A,B,mean(A),mean(B),diff,se,T,p-tukey,hedges
0,Adelie,Chinstrap,3700.662252,3733.088235,-32.425984,67.511684,-0.480302,0.880667,-0.073946
1,Adelie,Gentoo,3700.662252,5076.016260,-1375.354009,56.147971,-24.495169,0.000000,-2.860201
2,Chinstrap,Gentoo,3733.088235,5076.016260,-1342.928025,69.856928,-19.223978,0.000000,-2.875327


In [41]:
arr = np.array([
    [0, 0, 0],
    [1, 0, 2],
    [0, 3, 0]
])

arr == 0

array([[ True,  True,  True],
       [False,  True, False],
       [ True, False,  True]])